## Superstore Sales Analysis & Automated Reporting
**Goal:** Clean and validate sales data, find business insights, and generate
Excel reports automatically for any region.
**Tools:** Python, pandas, openpyxl

#### Step 1. Load Data
Loading the Superstore CSV file. `latin-1` encoding is used because the file
contains special characters.

In [11]:
import pandas as pd

df=pd.read_csv("data/Superstore.csv", encoding="latin-1")
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


#### 2. Data Inspection
Checking column types, missing values, duplicates and basic statistics
to understand the data before cleaning.

In [25]:
df.describe()
df.isnull().sum()
df.duplicated().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [41]:
print(df[["Order Date","Ship Date"]])

      Order Date   Ship Date
0     08-11-2016  11-11-2016
1     08-11-2016  11-11-2016
2     12-06-2016  16-06-2016
3     11-10-2015  18-10-2015
4     11-10-2015  18-10-2015
...          ...         ...
9989  21-01-2014  23-01-2014
9990  26-02-2017  03-03-2017
9991  26-02-2017  03-03-2017
9992  26-02-2017  03-03-2017
9993  04-05-2017  09-05-2017

[9994 rows x 2 columns]


In [45]:
print(df[df["Order Date"].isna()])

Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit]
Index: []

[0 rows x 21 columns]


In [103]:
print(df["Order Date"].dtype)


datetime64[ns]


#### 3. Data Cleaning
- Removed extra spaces from column names
- Converted Order Date and Ship Date from text to datetime (format dd-mm-yyyy)
- Removed duplicate rows

In [95]:
df.columns = df.columns.str.strip()
df["Order Date"] = pd.to_datetime(df["Order Date"], format= "%d-%m-%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format= "%d-%m-%Y")
df = df.drop_duplicates()

In [101]:
print(df[["Order Date", "Ship Date"]].head(5))

  Order Date  Ship Date
0 2016-11-08 2016-11-11
1 2016-11-08 2016-11-11
2 2016-06-12 2016-06-16
3 2015-10-11 2015-10-18
4 2015-10-11 2015-10-18


#### 4. Data Validation
Checks to confirm the data is reliable:
- Ship Date is never before Order Date
- Number of loss-making rows (negative profit)

In [109]:
assert (df["Ship Date"] >= df["Order Date"]).all()
print("negative sales", (df["Sales"]<0).sum())
print("Loss_making sale", (df["Profit"]<0).sum())

negative sales 0
Loss_making sale 1871


#### 5. Analysis
Sales and profit by region, monthly sales trend, top 10 customers,
and loss-making sub-categories.

In [199]:
# Highest sales and profit region
print("Highest sales and profit by region \n\n", df.groupby("Region")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False))

# Top 10 customers' share of total sales
top10 = df.groupby("Customer Name")["Sales"].sum().nlargest(10).sum()
print("\nTop 10 customers' share of total sales \n",round(top10 / df["Sales"].sum() * 100, 1), "%")

# Best and worst months
m = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum()
print("\n",m.idxmax(), m.max(),"  Best")
print(m.idxmin(), m.min(),"  Worst")

Highest sales and profit by region 

                Sales       Profit
Region                           
West     725457.8245  108418.4489
East     678781.2400   91522.7800
Central  501239.8908   39706.3625
South    391721.9050   46749.4303

Top 10 customers' share of total sales 
 6.7 %

 2017-11 118447.825   Best
2014-02 4519.892   Worst


In [157]:
import os
os.makedirs("outputs", exist_ok=True)

def generate_report(data, Region=None):
    d = data if Region is None else data [data["Region"] == Region]
    name = "All" if Region is None else Region

    kpis = pd.DataFrame({
        "KPI": ["Total Sales", "Total Profit", "Orders", "Profit Margin %"],
        "value":[
            round(d["Sales"].sum(), 2),
            round(d["Profit"].sum(), 2),
            d["Order ID"].nunique(),
            round(d["Profit"].sum() / d["Sales"].sum() * 100, 2)
        ],
    })
    by_cat = d.groupby("Category")[["Sales", "Profit"]].sum().round(2)
    by_region = d.groupby("Region")[["Sales", "Profit"]].sum().round(2)
    monthly = (d.groupby(d["Order Date"].dt.to_period("M").astype(str))["Sales"].sum().round(2))
    top_cust = (d.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False).head(10).round(2))

    file = f"outputs/sale_report_{name}.xlsx"
    with pd.ExcelWriter(file, engine="openpyxl") as w:
        kpis.to_excel(w, sheet_name="Summary", index=False)
        by_cat.to_excel(w, sheet_name=" By Category")
        by_region.to_excel(w, sheet_name=" By region")
        monthly.to_excel(w, sheet_name="Monthly Sales")
        top_cust.to_excel(w, sheet_name="Top Customers")
    print("saved", file)



#### 6. Automated Report
`generate_report(data, Region)` creates an Excel file with 4 sheets
(Summary, By Category, Monthly Sales, Top Customers).
Leave Region empty for all regions, or pass one like "West".
This lets the same code produce reports for any market.

In [159]:
generate_report(df)

saved outputs/sale_report_All.xlsx


In [161]:
generate_report(df, "West")

saved outputs/sale_report_West.xlsx


In [201]:
import os
print(os.getcwd())

C:\Users\anike\Python_Analysis
